# Introduction
We will use this notebook to explore the data initially, before initiating any trading algorithm.

# Import

In [ ]:
# Standard library imports
import datetime as dt
import os
import sys

# Third party imports
import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Local imports
from utilities import simple_returns, log_returns
from ar_script import rolling_ar_forecast
from ar_script import multiple_forecasts
from ar_script import multiple_scores
from ar_script import rolling_regression

# Data Analysis

In [ ]:
df_train = pd.read_csv('df_train.csv')

In [ ]:
df_train.head()

In [ ]:
df_train.tail()

In [ ]:
df_train.shape

In [ ]:
df_train.info()

In [ ]:
# Unique symbols
print('Number of unique symbols:', len(df_train['symbol'].unique()))
print('Number of unique dates:', len(df_train['date'].unique()))

In [ ]:
df_train_wide = df_train.pivot(index='date', values='close', columns='symbol')
df_train_wide.index = pd.to_datetime(df_train_wide.index)
df_train_wide = df_train_wide.sort_index()

In [ ]:
print("Total null values:", df_train_wide.isna().sum().sum())

In [ ]:
# Summary statistics
df_train_wide.describe()

In [ ]:
# Split into train-test sets
df_train_start = df_train_wide[df_train_wide.index < "2013-01-01"]
# Calculate log returns
df_train_log_returns = log_returns(df_train_start, 1, False)
# Calculate simple returns
df_train_simple_returns = simple_returns(df_train_start, 1, False)
df_train_log_returns.head(5)

In [ ]:
# Identify symbols
symbols = sorted(df_train['symbol'].unique().tolist())

In [ ]:
# Calculate forecasts for log returns
ar1_log_rets_forecasts = multiple_forecasts(df_train_log_returns, 
                                            symbols, AutoReg, 252, 
                                            1, lags=[1, 5, 21])
# Calculate R2 scores
ar1_log_rets_r2 = multiple_scores(ar1_log_rets_forecasts, 
                                  symbols, r2_score,'r2')


In [19]:
ar1_log_rets_r2[ar1_log_rets_r2['r2'] > 0].shape[0]

22

In [ ]:
# Calculate AR(1) forecasts for simple returns
ar1_simple_rets_forecasts = multiple_forecasts(df_train_simple_returns, 
                                               symbols, AutoReg, 252, 
                                               1, lags=[1, 5, 21])
# Calculate R2 scores
ar1_simple_rets_r2 = multiple_scores(ar1_simple_rets_forecasts, 
                                     symbols, r2_score, 'r2')

In [20]:
ar1_simple_rets_r2[ar1_simple_rets_r2['r2'] > 0].shape[0]

24

,forecast,ACTS
date,,
2011-01-04,0.000772,-0.064103
2011-01-05,-0.002936,-0.022562
2011-01-06,0.000628,-0.018961
2011-01-07,-0.000391,0.004202
2011-01-10,-0.001370,-0.030962
...,...,...
2012-12-24,0.005013,-0.011519
2012-12-26,0.003286,0.010925
2012-12-27,-0.003552,-0.013689
